# 06.10 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/10-recommenders/solutions.ipynb)

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

M = pd.DataFrame(
    [[1, 1, 0, 1, 0],
     [1, 1, 0, 0, 0],
     [0, 1, 1, 0, 1],
     [1, 0, 0, 1, 1]],
    index=["Ana", "Ben", "Chi", "Dev"],
    columns=["mug", "teapot", "candle", "coaster", "lantern"])
M

,mug,teapot,candle,coaster,lantern
Ana,1,1,0,1,0
Ben,1,1,0,0,0
Chi,0,1,1,0,1
Dev,1,0,0,1,1


In [2]:
from sklearn.metrics.pairwise import cosine_similarity
item_sim = pd.DataFrame(cosine_similarity(M.T), index=M.columns, columns=M.columns)

## Exercise 1

From the lesson's table:

- mug: sim(teapot, mug) + sim(candle, mug) + sim(lantern, mug) = 0.667 + 0 + 0.408 = **1.075**
- coaster: 0.408 + 0 + 0.500 = **0.908**

Recommend the **mug**.

In [3]:
owned = M.loc["Chi"] == 1
print(item_sim.loc[owned[owned].index].sum()[~owned].sort_values(ascending=False).round(3))

mug        1.075
coaster    0.908
dtype: float64


## Exercise 2

In [4]:
user_sim = pd.DataFrame(cosine_similarity(M), index=M.index, columns=M.index)
print(user_sim.round(3))
nearest = user_sim["Ben"].drop("Ben").idxmax()
suggest = M.columns[(M.loc[nearest] == 1) & (M.loc["Ben"] == 0)]
print(f"\nBen's most similar customer: {nearest}; recommend {list(suggest)}")

       Ana    Ben    Chi    Dev
Ana  1.000  0.816  0.333  0.667
Ben  0.816  1.000  0.408  0.408
Chi  0.333  0.408  1.000  0.333
Dev  0.667  0.408  0.333  1.000

Ben's most similar customer: Ana; recommend ['coaster']


Ben's nearest customer is Ana (cosine 0.816), who also bought a coaster, so user-user gives the **same** recommendation here. With real data the two can differ: item-item is usually preferred in shops because product similarities change slowly and can be precomputed, while customers come and go.

## Exercise 3

In [5]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"], **{"Customer ID": sales["Customer ID"].astype(int)})
print(f"{len(sales):,} clean transaction lines, {sales['Customer ID'].nunique():,} customers")

392,693 clean transaction lines, 4,338 customers


In [6]:
# Hold out each customer's LAST invoice; learn from everything before it.
last_invoice = sales.sort_values("InvoiceDate").groupby("Customer ID")["Invoice"].last()
is_last = sales["Invoice"].isin(set(last_invoice))
train, test = sales[~is_last], sales[is_last]

top_products = train["StockCode"].value_counts().head(500).index
tr = train[train["StockCode"].isin(top_products)]
R = pd.crosstab(tr["Customer ID"], tr["StockCode"]).clip(upper=1)     # 1 = bought at least once
names = sales.drop_duplicates("StockCode").set_index("StockCode")["Description"]
print(f"train: {R.shape[0]:,} customers x {R.shape[1]} products, {R.values.mean():.1%} of cells are 1")

from sklearn.metrics.pairwise import cosine_similarity
S = cosine_similarity(R.T.values)          # product x product
np.fill_diagonal(S, 0)                     # a product is not its own recommendation
sim = pd.DataFrame(S, index=R.columns, columns=R.columns)

train: 2,805 customers x 500 products, 6.8% of cells are 1


In [7]:
truth = test[test["StockCode"].isin(top_products)].groupby("Customer ID")["StockCode"].apply(set)
customers = [c for c in R.index if c in truth.index]
products = np.array(R.columns)
bought = R.values > 0
popularity = R.sum().values.astype(float)

def hit_rate(score_matrix, k=10):
    hits = 0
    for c in customers:
        i = R.index.get_loc(c)
        s = score_matrix[i].copy(); s[bought[i]] = -np.inf          # never recommend what they own
        top_k = products[np.argsort(-s)[:k]]
        hits += bool(truth[c] & set(top_k))
    return hits / len(customers)

cf_scores = R.values @ S                                   # sum of similarities to what they bought
pop_scores = np.tile(popularity, (len(R), 1))              # same list for everyone
print(f"customers evaluated: {len(customers):,}")

for k in [1, 5, 10, 20]:
    p, c = hit_rate(pop_scores, k), hit_rate(cf_scores, k)
    print(f"k = {k:>2}: popularity {p:.1%}   collaborative {c:.1%}   ratio {c / p:.2f}")

customers evaluated: 2,631
k =  1: popularity 3.5%   collaborative 7.2%   ratio 2.08


k =  5: popularity 12.2%   collaborative 20.5%   ratio 1.68


k = 10: popularity 20.3%   collaborative 30.0%   ratio 1.48
k = 20: popularity 32.4%   collaborative 40.9%   ratio 1.26


Both hit rates rise with k, since longer lists have more chances, and collaborative filtering stays ahead at every k. Its advantage is **largest for short lists**: at k = 1 it gets about twice as many hits as best-sellers (7.2% vs 3.5%), at k = 20 only about a quarter more (40.9% vs 32.4%). Long lists fill up with popular items anyway, so the two methods converge. Short lists are exactly where it matters: a "you may also like" strip shows three or four products.